In [1]:
import mlflow
import polars as pl
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score
from torch.utils.data import DataLoader, Dataset
from torchinfo import summary
from tqdm.notebook import tqdm
from transformers import AutoModel, AutoTokenizer
from transformers.models.bert.modeling_bert import BertModel
from mlflow.models import infer_signature

# Types
from transformers.models.bert.tokenization_bert_fast import BertTokenizerFast

import matplotlib.pyplot as plt
from src.config import MPL_STYLE_DIR, PROCESSED_DATA_DIR
from src.db import PBWarehouse
from src.models import Tweet

mlflow.set_tracking_uri("http://192.168.100.203:5000")
mlflow.set_experiment("[CAPSTONE-2] hatebert-finetuning")

warehouse = PBWarehouse()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

2025-07-07 14:24:24.098 | INFO     | src.config:<module>:26 - Loaded environment variables from /home/iragca/Documents/github/capstone-project-2/.env
2025-07-07 14:24:24.098 | INFO     | src.config:<module>:60 - PROJECT_ROOT: /home/iragca/Documents/github/capstone-project-2
2025-07-07 14:24:24.099 | INFO     | src.config:<module>:61 - DATA_DIR: /home/iragca/Documents/github/capstone-project-2/data


device(type='cuda')

In [2]:
data = warehouse.client.collection("tweets_v2").get_full_list(
    query_params={
        "filter": (
            "has_blm_hashtag = true || "
            "is_reply_to_blm = true && "
            "creation_date >= '2020-03-26' "
            "&& creation_date <= '2020-07-24' "
            "&& language = 'en'",
        )
    }
)
tweets: list[Tweet] = [Tweet(**r.__dict__) for r in data]

In [3]:
df = pl.DataFrame(
    [t.model_dump() for t in tweets],
    schema={
        "tweet_id": pl.Utf8,
        "text": pl.Utf8,
        "status_link": pl.Utf8,
        "user_id": pl.Utf8,
        "is_extremist": pl.Boolean,
        "is_annotated": pl.Boolean,
        "in_reply_to_status_link": pl.Utf8,
        "in_reply_to_status_id": pl.Utf8,
        "bookmark_count": pl.Int64,
        "views": pl.Int64,
        "retweet_count": pl.Int64,
        "favorite_count": pl.Int64,
        "reply_count": pl.Int64,
        "quote_count": pl.Int64,
        "conversation_id": pl.Utf8,
        "retweet_tweet_id": pl.Utf8,
        "quoted_status_id": pl.Utf8,
        "community_note": pl.Utf8,
        "language": pl.Utf8,
        "source": pl.Utf8,
        "creation_date": pl.Utf8,
        "has_blm_hashtag": pl.Boolean,
        "fetched_replies": pl.Boolean,
        "is_reply_to_blm": pl.Boolean,
    },
).with_columns(
    pl.col("creation_date").str.strptime(pl.Datetime, format="%Y-%m-%d %H:%M:%S%.3fZ")
)
df

tweet_id,text,status_link,user_id,is_extremist,is_annotated,in_reply_to_status_link,in_reply_to_status_id,bookmark_count,views,retweet_count,favorite_count,reply_count,quote_count,conversation_id,retweet_tweet_id,quoted_status_id,community_note,language,source,creation_date,has_blm_hashtag,fetched_replies,is_reply_to_blm
str,str,str,str,bool,bool,str,str,i64,i64,i64,i64,i64,i64,str,str,str,str,str,str,datetime[ms],bool,bool,bool
"""1286447415080312833""","""Heyhey if you are someone/know…","""https://x.com/ChuckMockler/sta…","""347913343""",false,false,"""""","""""",0,0,1,2,0,0,"""1286447415080312833""","""""","""""","""""","""en""","""Twitter for iPhone""",2020-07-23 23:45:39,true,true,false
"""1286440938106163203""","""Thank you @VoteAdamMedrano for…","""https://x.com/AdamBazaldua/sta…","""833871911272722433""",false,false,"""""","""""",0,0,4,16,4,1,"""1286440938106163203""","""""","""""","""""","""en""","""Twitter for iPhone""",2020-07-23 23:19:55,true,true,false
"""1286450496329285632""","""I have ten-ish years of experi…","""https://x.com/LilyShumarKray/s…","""773036078""",false,false,"""https://x.com/LilyShumarKray/s…","""1286450494349533184""",0,0,0,3,0,0,"""1286450494349533184""","""""","""""","""""","""en""","""""",2020-07-23 23:57:54,false,true,true
"""1286449764507291650""","""Opening day is underway for @m…","""https://x.com/HarrisWarRoom/st…","""1148212967332204544""",false,false,"""""","""""",0,0,10,50,0,1,"""1286449764507291650""","""""","""""","""""","""en""","""Twitter Web App""",2020-07-23 23:54:59,true,true,false
"""1286439076233650177""","""People are either being Dof an…","""https://x.com/SoliPhilander/st…","""108571906""",false,false,"""""","""""",0,0,0,7,0,0,"""1286439076233650177""","""""","""""","""""","""en""","""Twitter for Android""",2020-07-23 23:12:31,true,true,false
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""1275947325983244288""","""@johncardillo This is such BS.…","""https://x.com/MainStreetMuse/s…","""179785566""",false,false,"""https://x.com/johncardillo/sta…","""1275946651308392450""",0,0,0,2,0,0,"""1275946651308392450""","""""","""""","""""","""en""","""""",2020-06-25 00:22:03,false,false,true
"""1269056990178938880""","""@rebeinstein ... the only good…","""https://x.com/alanekennedylaw/…","""2496399067""",false,false,"""https://x.com/rebeinstein/stat…","""1269055716972564481""",0,0,0,3,1,0,"""1269055716972564481""","""""","""""","""""","""en""","""""",2020-06-06 00:02:19,false,false,true
"""1273000292720668672""","""@tomselliott @dbongino @thread…","""https://x.com/MSMCali/status/1…","""728434941843804161""",false,false,"""https://x.com/tomselliott/stat…","""1272993765670739969""",0,0,0,1,1,0,"""1272986923951407106""","""""","""""","""""","""en""","""""",2020-06-16 21:11:35,false,false,true


In [4]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "Hate-speech-CNERG/bert-base-uncased-hatexplain"  # HATEBERT on Hugging Face

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)




In [5]:
inputs = tokenizer("I hate you", return_tensors="pt")
outputs = model(**inputs)

logits = outputs.logits

probs = torch.nn.functional.softmax(logits, dim=-1)
predicted_class = torch.argmax(probs, dim=-1)

print(f"Logits: {logits}")
print(f"Probabilities: {probs}")
print(f"Predicted class: {predicted_class.item()}")

BertSdpaSelfAttention is used but `torch.nn.functional.scaled_dot_product_attention` does not support non-absolute `position_embedding_type` or `output_attentions=True` or `head_mask`. Falling back to the manual attention implementation, but specifying the manual implementation will be required from Transformers version v5.0.0 onwards. This warning can be removed using the argument `attn_implementation="eager"` when loading the model.


Logits: tensor([[-1.2805,  0.6910, -0.1503]], grad_fn=<AddmmBackward0>)
Probabilities: tensor([[0.0887, 0.6368, 0.2745]], grad_fn=<SoftmaxBackward0>)
Predicted class: 1


In [9]:
test_df = df.select(
    [
        pl.col("text"),
    ]
).with_columns(pl.lit(None).alias("is_hateful").cast(pl.Int8))
test_df

text,is_hateful
str,i8
"""Heyhey if you are someone/know…",null
"""Thank you @VoteAdamMedrano for…",null
"""I have ten-ish years of experi…",null
"""Opening day is underway for @m…",null
"""People are either being Dof an…",null
…,…
"""@johncardillo This is such BS.…",null
"""@rebeinstein ... the only good…",null
"""@tomselliott @dbongino @thread…",null


In [10]:
def classify_text(text: str) -> int:
    inputs = tokenizer(text, return_tensors="pt")
    outputs = model(**inputs)
    logits = outputs.logits
    probs = torch.nn.functional.softmax(logits, dim=-1)
    predicted_class = torch.argmax(probs, dim=-1)
    return predicted_class.item()


In [11]:
classify_text("I hate you")

1

In [12]:
test_df = df.with_columns(
    pl.col("text").map_elements(lambda x: classify_text(x), return_dtype=pl.Int8).alias("is_hateful")
)

In [18]:
# test_df.write_csv(PROCESSED_DATA_DIR / "hatebert-test.csv")
test_df = pl.read_csv(PROCESSED_DATA_DIR / "hatebert-test.csv")

In [19]:
test_df

tweet_id,text,status_link,user_id,is_extremist,is_annotated,in_reply_to_status_link,in_reply_to_status_id,bookmark_count,views,retweet_count,favorite_count,reply_count,quote_count,conversation_id,retweet_tweet_id,quoted_status_id,community_note,language,source,creation_date,has_blm_hashtag,fetched_replies,is_reply_to_blm,is_hateful
i64,str,str,i64,bool,bool,str,str,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,str,str,bool,bool,bool,i64
1286447415080312833,"""Heyhey if you are someone/know…","""https://x.com/ChuckMockler/sta…",347913343,false,false,"""""","""""",0,0,1,2,0,0,1286447415080312833,"""""","""""","""""","""en""","""Twitter for iPhone""","""2020-07-23T23:45:39.000""",true,true,false,1
1286440938106163203,"""Thank you @VoteAdamMedrano for…","""https://x.com/AdamBazaldua/sta…",833871911272722433,false,false,"""""","""""",0,0,4,16,4,1,1286440938106163203,"""""","""""","""""","""en""","""Twitter for iPhone""","""2020-07-23T23:19:55.000""",true,true,false,1
1286450496329285632,"""I have ten-ish years of experi…","""https://x.com/LilyShumarKray/s…",773036078,false,false,"""https://x.com/LilyShumarKray/s…","""1286450494349533184""",0,0,0,3,0,0,1286450494349533184,"""""","""""","""""","""en""","""""","""2020-07-23T23:57:54.000""",false,true,true,1
1286449764507291650,"""Opening day is underway for @m…","""https://x.com/HarrisWarRoom/st…",1148212967332204544,false,false,"""""","""""",0,0,10,50,0,1,1286449764507291650,"""""","""""","""""","""en""","""Twitter Web App""","""2020-07-23T23:54:59.000""",true,true,false,1
1286439076233650177,"""People are either being Dof an…","""https://x.com/SoliPhilander/st…",108571906,false,false,"""""","""""",0,0,0,7,0,0,1286439076233650177,"""""","""""","""""","""en""","""Twitter for Android""","""2020-07-23T23:12:31.000""",true,true,false,1
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
1275947325983244288,"""@johncardillo This is such BS.…","""https://x.com/MainStreetMuse/s…",179785566,false,false,"""https://x.com/johncardillo/sta…","""1275946651308392450""",0,0,0,2,0,0,1275946651308392450,"""""","""""","""""","""en""","""""","""2020-06-25T00:22:03.000""",false,false,true,1
1269056990178938880,"""@rebeinstein ... the only good…","""https://x.com/alanekennedylaw/…",2496399067,false,false,"""https://x.com/rebeinstein/stat…","""1269055716972564481""",0,0,0,3,1,0,1269055716972564481,"""""","""""","""""","""en""","""""","""2020-06-06T00:02:19.000""",false,false,true,1
1273000292720668672,"""@tomselliott @dbongino @thread…","""https://x.com/MSMCali/status/1…",728434941843804161,false,false,"""https://x.com/tomselliott/stat…","""1272993765670739969""",0,0,0,1,1,0,1272986923951407106,"""""","""""","""""","""en""","""""","""2020-06-16T21:11:35.000""",false,false,true,1


In [4]:
test_df.filter(pl.col("is_hateful") == 2).select(
    [pl.col("text"), pl.col("is_hateful")],
).to_pandas()

,text,is_hateful
0,@fuzzyfromyt goat,2
1,Essential workers can’t even work because of t...,2
2,Closet racists came out real quick after the n...,2
3,I challenge anyone to click on BLM Donate Butt...,2
4,@s_butler2015 @realDonaldTrump Coach go and te...,2
...,...,...
870,@nbstv The apology came because the outcome wa...,2
871,@AamerAnwar @SNP_Porty_Craig Absolute lowlife ...,2
872,@johncardillo @gatewaypundit #BlackLivesMatter...,2
873,@ScotsmanGrumpy The gaming community will neve...,2


In [5]:
test_df.filter(pl.col("is_hateful") == 1).select(
    [pl.col("text"), pl.col("is_hateful")],
).to_pandas()

,text,is_hateful
0,Heyhey if you are someone/know anyone running ...,1
1,Thank you @VoteAdamMedrano for taking the lead...,1
2,I have ten-ish years of experience as a martia...,1
3,Opening day is underway for @mlb and their log...,1
4,"People are either being Dof and Deliberate, or...",1
...,...,...
41475,@johncardillo This is such BS. They want to be...,1
41476,@rebeinstein ... the only good cops are no cop...,1
41477,@tomselliott @dbongino @threadreaderapp please...,1
41478,@HackneyAbbott @socialistcam This is starting ...,1


In [6]:
def has_hateful_words(text: str) -> bool:

    terms = [
        "bnwo",
        "bbc",
        "blacksupremacy",
        "queenofspades",
        "blacked",
        "qos"
    ]

    return any(term in text.lower() for term in terms)

hate_df = test_df.with_columns(
    pl.col("text").map_elements(lambda x: has_hateful_words(x), return_dtype=pl.Boolean).alias("has_hateful_words")
)


In [ ]:
hate_df["has_hateful_words"]

has_hateful_words,count
bool,u32
true,590
false,41845


In [22]:
hate_df = test_df.with_columns(
    pl.col("creation_date").str.strptime(pl.Datetime, format="%Y-%m-%d %H:%M:%S%.3fZ")  
    )

InvalidOperationError: conversion from `str` to `datetime[ms]` failed in column 'creation_date' for 226 out of 226 values: ["2020-06-03T14:19:15.000", "2020-06-12T15:06:50.000", … "2020-06-19T19:08:21.000"]

You might want to try:
- setting `strict=False` to set values that cannot be converted to `null`
- using `str.strptime`, `str.to_date`, or `str.to_datetime` and providing a format string

In [21]:
hate_df

tweet_id,text,status_link,user_id,is_extremist,is_annotated,in_reply_to_status_link,in_reply_to_status_id,bookmark_count,views,retweet_count,favorite_count,reply_count,quote_count,conversation_id,retweet_tweet_id,quoted_status_id,community_note,language,source,creation_date,has_blm_hashtag,fetched_replies,is_reply_to_blm,is_hateful
i64,str,str,i64,bool,bool,str,str,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,str,datetime[ms],bool,bool,bool,i64
1286447415080312833,"""Heyhey if you are someone/know…","""https://x.com/ChuckMockler/sta…",347913343,false,false,"""""","""""",0,0,1,2,0,0,1286447415080312833,"""""","""""","""""","""en""","""Twitter for iPhone""",null,true,true,false,1
1286440938106163203,"""Thank you @VoteAdamMedrano for…","""https://x.com/AdamBazaldua/sta…",833871911272722433,false,false,"""""","""""",0,0,4,16,4,1,1286440938106163203,"""""","""""","""""","""en""","""Twitter for iPhone""",null,true,true,false,1
1286450496329285632,"""I have ten-ish years of experi…","""https://x.com/LilyShumarKray/s…",773036078,false,false,"""https://x.com/LilyShumarKray/s…","""1286450494349533184""",0,0,0,3,0,0,1286450494349533184,"""""","""""","""""","""en""","""""",null,false,true,true,1
1286449764507291650,"""Opening day is underway for @m…","""https://x.com/HarrisWarRoom/st…",1148212967332204544,false,false,"""""","""""",0,0,10,50,0,1,1286449764507291650,"""""","""""","""""","""en""","""Twitter Web App""",null,true,true,false,1
1286439076233650177,"""People are either being Dof an…","""https://x.com/SoliPhilander/st…",108571906,false,false,"""""","""""",0,0,0,7,0,0,1286439076233650177,"""""","""""","""""","""en""","""Twitter for Android""",null,true,true,false,1
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
1275947325983244288,"""@johncardillo This is such BS.…","""https://x.com/MainStreetMuse/s…",179785566,false,false,"""https://x.com/johncardillo/sta…","""1275946651308392450""",0,0,0,2,0,0,1275946651308392450,"""""","""""","""""","""en""","""""",null,false,false,true,1
1269056990178938880,"""@rebeinstein ... the only good…","""https://x.com/alanekennedylaw/…",2496399067,false,false,"""https://x.com/rebeinstein/stat…","""1269055716972564481""",0,0,0,3,1,0,1269055716972564481,"""""","""""","""""","""en""","""""",null,false,false,true,1
1273000292720668672,"""@tomselliott @dbongino @thread…","""https://x.com/MSMCali/status/1…",728434941843804161,false,false,"""https://x.com/tomselliott/stat…","""1272993765670739969""",0,0,0,1,1,0,1272986923951407106,"""""","""""","""""","""en""","""""",null,false,false,true,1


In [15]:
sorted_df = hate_df.filter(pl.col("has_hateful_words")).sort(pl.col("creation_date")).group_by_dynamic("creation_date", every="1m").agg(
    [
        pl.col("tweet_id").count().alias("total_tweets"),
    ])

ComputeError: null values in dynamic group_by not supported, fill nulls.